- Import all modules

In [25]:
# Import libraries:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

- Load the trained models

In [26]:
# Load all the models: Model, Scaler, OHE
model = load_model('model.h5')

# Load encoder and scaler
with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

- Test Data

In [27]:
# Example input data:
input_data = {
    'CreditScore' : 600,
    'Geography' : 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure' : 3,
    'Balance' : 60000,
    'NumOfProducts' : 2,
    'HasCrCard' : 1,
    'IsActiveMember' : 1,
    'EstimatedSalary' : 50000
}

In [28]:
# Convert input_data into DataFrame:
data = pd.DataFrame([input_data])
# Encode Geography:
encode_geo = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_feature_names = onehot_encoder_geo.get_feature_names_out(['Geography'])
# Encode Gender:
data['Gender'] = label_encoder_gender.transform(data['Gender'])
# Drop Geography and add encoded Geography:
data[geo_feature_names] = encode_geo
data = data.drop('Geography', axis=1)
data

c:\Users\hosha\OneDrive\Work\UpSkilling\GenAI\Section 13. ANN Classification Project\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [29]:
# Scale the data:
input_scaled = scaler.transform(data)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

- Predict on scaled unseen data

In [31]:
# Predict using the model:
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 548ms/step


array([[0.02455345]], dtype=float32)

- Get prediction probabilities

In [33]:
# Probabilities:
prediction_proba = prediction[0][0]
print(prediction_proba)

0.024553452


In [34]:
# Conditional logic:
if prediction_proba > 0.5:
    print('The customer is likely to churn')
else:
    print('The customer is not likely to churn')

The customer is not likely to churn


In [79]:
def predict_data(data, ohe_columns = None, label_columns = None):
    if isinstance(data, dict):
        data = [data]
    elif isinstance(data, pd.DataFrame):
        data = data.to_dict(orient='records')
    elif isinstance(data, list):
        pass
    else:
        raise ValueError(f"Unsupported input type:{type(data)}. Expected dict, list or DataFrame")

    results = []
    for d in data:
        df = pd.DataFrame([d])

        # Dynamic OHE:
        if ohe_columns:
            for col, encoder in ohe_columns.items():
                encoded = encoder.transform([[df[col].values[0]]]).toarray()
                feature_names = encoder.get_feature_names_out([col])
                df[feature_names] = encoded
                df = df.drop(col, axis=1)

        # Dynamic Label:
        if label_columns:
            for col, encoder in label_columns.items():
                df[col] = encoder.transform(df[col])

        # Scale and Predict:
        df_scaled = scaler.transform(df)
        prediction = model.predict(df_scaled)
        prediction_proba = prediction[0][0]

        # Result:
        result = 'Likely to churn' if prediction_proba > 0.5 else 'Not likely to churn'
        results.append({'Probability' : round(float(prediction_proba), 4), 'result' : result})
    
    for i, r in enumerate(results):
        print(f"Sample {i+1}: {r['result']:<25} | Churn Probability: {r['Probability']:.2%}")

In [80]:
samples = [
    {
        'CreditScore': 720,
        'Geography': 'Germany',
        'Gender': 'Female',
        'Age': 35,
        'Tenure': 5,
        'Balance': 120000,
        'NumOfProducts': 1,
        'HasCrCard': 1,
        'IsActiveMember': 0,
        'EstimatedSalary': 80000
    },
    {
        'CreditScore': 450,
        'Geography': 'Spain',
        'Gender': 'Male',
        'Age': 52,
        'Tenure': 8,
        'Balance': 0,
        'NumOfProducts': 1,
        'HasCrCard': 0,
        'IsActiveMember': 0,
        'EstimatedSalary': 30000
    },
    {
        'CreditScore': 800,
        'Geography': 'France',
        'Gender': 'Female',
        'Age': 28,
        'Tenure': 2,
        'Balance': 45000,
        'NumOfProducts': 3,
        'HasCrCard': 1,
        'IsActiveMember': 1,
        'EstimatedSalary': 95000
    },
    {
        'CreditScore': 580,
        'Geography': 'Germany',
        'Gender': 'Male',
        'Age': 45,
        'Tenure': 7,
        'Balance': 150000,
        'NumOfProducts': 2,
        'HasCrCard': 0,
        'IsActiveMember': 1,
        'EstimatedSalary': 62000
    },
    {
        'CreditScore': 670,
        'Geography': 'Spain',
        'Gender': 'Female',
        'Age': 31,
        'Tenure': 1,
        'Balance': 75000,
        'NumOfProducts': 2,
        'HasCrCard': 1,
        'IsActiveMember': 0,
        'EstimatedSalary': 110000
    }
]

ohe_col = {'Geography' : onehot_encoder_geo}
lab_col = {'Gender' : label_encoder_gender}

# Check func:
predict_data(samples, ohe_columns=ohe_col, label_columns=lab_col)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
Sample 1: Not likely to churn       | Churn Probability: 36.19%
Sample 2: Likely to churn           | Churn Probability: 62.02%
Sample 3: Not likely to churn       | Churn Probability: 46.28%
Sample 4: Not likely to churn       | Churn Probability: 16.68%
Sample 5: Not likely to churn       | Churn Probability: 13.22%
